In [2]:
import pandas as pd 
import numpy as np
from pathlib import Path
from time import time
from zipfile import ZipFile
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier,
                              AdaBoostClassifier)
from sklearn.preprocessing import (StandardScaler,
                                   OneHotEncoder)
# from sklearn.preprocessing import spl
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)


In [ ]:
data_path = Path("data")
file_path = Path("data/home-credit-default-risk.zip")
file_name = "home-credit-default-risk"
data_path = data_path/file_name
data_path.mkdir(parents=True,exist_ok=True)
if file_path.exists():
    with ZipFile(file_path,"r") as f:
        f.extractall(data_path)
        print("Done")
else:
    print("[ERROR] File Doesn't Exist.")

## cheching files
Required_file = [
    "bureau.csv",
    "bureau_balance.csv",
    "credit_card_balance.csv",
    "installments_payments.csv",
    "POS_CASH_balance.csv",    
    "previous_application.csv"
]


In [ ]:
application = (data_path/"application_train.csv")
df = pd.read_csv(application)

n_rows , n_columns = df.shape

numeric_data = df.select_dtypes(include=["number"]).columns
categorical_data = df.select_dtypes(include=["str"]).columns

print(f"DATAFRAME :{application.name} size :{(application.stat().st_size) / 1024**2:.0f} MB")
print(f"{n_rows} ROWS x {n_columns} COLUMNS")
print(f"NUMERIC : {len(numeric_data)}")
print(f"CATOGORICAL : {len(categorical_data)}")
print(f"SK_ID_CURR : {df.SK_ID_CURR.nunique()} "
      f"(unique : {df.SK_ID_CURR.is_unique})")

In [ ]:
TARGET = df["TARGET"]
positive_rate = TARGET.mean()
imbalance = (1-positive_rate)/positive_rate

count = TARGET.value_counts().sort_index()
print("Target Distribution")
print(f"On_time payment        : {count[0]:>7,}    {(1-positive_rate)*100:5>.2f}%")
print(f"delayed                : {count[1]:>7,}    {positive_rate*100:5>.2f}% ")
print(f"imablance (ratio diff) : {imbalance:5>.2f}%")

Dummy = DummyClassifier(strategy="most_frequent").fit(df["AMT_CREDIT"],TARGET)
predict_prob = Dummy.predict_proba(df["AMT_CREDIT"])
acc = Dummy.predict(df["AMT_CREDIT"])
rou = Dummy.predict_proba(df["AMT_CREDIT"])[:,0]
print(f"Accuracy      : {accuracy_score(TARGET,acc):>6.2f}")
print(f"rou_auc_score : {roc_auc_score(TARGET,rou):>6.2f}")



In [ ]:
missing_value = df.isnull().mean()
any_missing_value = (missing_value > 0).sum()
more_than_50per_missing_value =(missing_value > 0.5).sum()

print(f"Missing value in {any_missing_value} colmn out of {n_columns} colmn")
print(f"missing_value colmn more than 50% {more_than_50per_missing_value}")
print((missing_value[missing_value>0]
       .sort_values(ascending=False)
       .head(10)
       .mul(100)
       .to_frame()
       .rename(columns={0:"missing_percent"})
       .round(4)
       ))


In [ ]:
installments_payments_agg = {
    "AMT_PAYMENT" : "sum",
    "AMT_INSTALMENT" : "sum",
    "NUM_INSTALMENT_NUMBER" : "max"
}
previous_application_agg = {
    'AMT_ANNUITY' :"sum",
    'AMT_APPLICATION':"mean",
    'AMT_CREDIT':"mean",
    'AMT_DOWN_PAYMENT':"sum",
    'AMT_GOODS_PRICE':"sum",
    'NFLAG_LAST_APPL_IN_DAY':"sum",
    'NFLAG_MICRO_CASH':"mean",
    'RATE_DOWN_PAYMENT':"sum",
    'RATE_INTEREST_PRIMARY':"mean",
    'RATE_INTEREST_PRIVILEGED':"mean",
    'CNT_PAYMENT':"sum",
    'DAYS_FIRST_DRAWING':"mean",
    'DAYS_FIRST_DUE':"mean",
    'DAYS_LAST_DUE_1ST_VERSION':"mean",
    'DAYS_LAST_DUE':"mean",
    'DAYS_TERMINATION':"mean",
    'NFLAG_INSURED_ON_APPROVAL':"sum",    
}
credit_card_balance_agg = {
    'MONTHS_BALANCE':"mean",
    'AMT_BALANCE':"sum",
    'AMT_CREDIT_LIMIT_ACTUAL':"mean",
    'AMT_PAYMENT_CURRENT':"sum",
    'AMT_PAYMENT_TOTAL_CURRENT':"sum",
    'AMT_RECEIVABLE_PRINCIPAL':"mean"
}
bureau_agg={
    'DAYS_CREDIT':"max",
    'CREDIT_DAY_OVERDUE':"max",
    'AMT_CREDIT_MAX_OVERDUE':"max",
    'AMT_CREDIT_SUM':"mean",
    'AMT_CREDIT_SUM_DEBT':"mean",
    'AMT_CREDIT_SUM_LIMIT':"mean",
    'AMT_CREDIT_SUM_OVERDUE':"maen",
    'AMT_ANNUITY':"mean"
}
POS_CASH_balance_agg={
    'MONTHS_BALANCE':"mean",
    'CNT_INSTALMENT':"mean",
    'CNT_INSTALMENT_FUTURE':"sum"
}

In [ ]:
data_dict = {}
for file_name in Required_file:
    df = pd.read_csv(data_path/file_name)
    clean_name = file_name.removesuffix(".csv")
    data_dict[file_name] = df.copy()
    print(file_name)

In [ ]:
data_dict.pop("bureau.csv",None)
data_dict.pop("bureau_balance.csv",None)
data_dict.pop("application_train.csv",None)
data_dict.pop("application_test.csv",None)
data_dict[""]

In [ ]:
def aggregating_num_data(df:pd.DataFrame,
                         id_colmn : str,
                         df_agg:dict,
                         cat_colm:list,
                         cat_agg_param:dict):
    final_df = df.copy()
    num_df = final_df.select_dtypes(include="number")
    cat_df = final_df.select_dtypes(include="str")
    num_colm_df = num_df.columns.to_list()
    cat_colm_df = cat_df.columns.to_list()


    if num_colm_df:
        agg_data = df.groupby(id_colmn).agg(df_agg)
        agg_data.columns = [f"{c}_{s}".upper() for c,s in agg_data.columns]
        agg_data.reset_index(inplace=True)

    if cat_colm_df:
        dummies = pd.get_dummies(cat_df[cat_colm])
        dummies[id_colmn] = df[id_colmn]
        cat_agg = dummies.groupby(id_colmn).agg(cat_agg_param)
        cat_agg.columns = [f"{c}_{s}".upper() for c,s in cat_agg.columns]
        cat_agg.reset_index(inplace=True)
        agg_data = agg_data.merge(cat_agg,how="left",on=id_colmn)

    return agg_data
    

def merging_df(main_df:pd.DataFrame,
               agg_df,
               by:str):
    final_df = main_df.copy()
    if isinstance(agg_df,list):
        for i in agg_df:
            final_df = main_df.merge(i,on=by,how="left")
    else:
        final_df = main_df.merge(agg_df,on=by,how="left")

        return final_df

In [ ]:
new_df = pd.DataFrame()

In [ ]:
bureau_df = merging_df(data_dict["bureau.csv"],data_dict["bureau_balance.csv"],by="SK_ID_BUREAU")

In [ ]:
df_path = Path(r"D:\Projects\Loan-Default-Predictor\data_maked")

In [ ]:
merging_df()

In [ ]:
bureau_df.shape

In [ ]:
import pyarrow

In [ ]:
bureau_df.to_csv(df_path/"bureau_grouped.csv",index=False)

In [ ]:
bureau_df

In [ ]:
bureau_df.columns

In [3]:
df = pd.read_csv(r"data\home-credit-default-risk\application_train.csv")
from sklearn.model_selection import train_test_split
X = df.drop(["TARGET"],axis=1)
y = df.TARGET
x_train, x_test, y_train, y_test = train_test_split(X,y,train_size=0.95)

In [ ]:
yy = (df.corr(numeric_only=True)["TARGET"])
aaa = yy[yy.abs() < 0.02].index.to_list()
yy,aaa

In [ ]:
aaa

In [ ]:
(df["DAYS_EMPLOYED"] >25550).replace(value=np.nan,)

In [ ]:
yy.to_list()

In [ ]:
y_train.head(10)

In [8]:
numeric_data = Pipeline(steps=[
    ("SimpleImputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())]
)
cat_data = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent",fill_value="missing")),
    ("one",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])
column_Transformer = ColumnTransformer(transformers=[
    ("num",numeric_data,x_train.select_dtypes(include="number").columns.to_list()),
    ("cat",cat_data,x_train.select_dtypes(include="str").columns.to_list())
])

whole_process = Pipeline(steps=[
    ("Column_Transformer",column_Transformer),
    ("model",RandomForestClassifier(n_estimators=200))
])
model = whole_process.fit(x_train,y_train)
model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Column_Transformer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [ ]:
a = model.predict(x_test)
ss = accuracy_score(y_test,a)
aa = model.predict_proba(x_test)[:,1]
sss = roc_auc_score(y_test,aa)
#some change need


In [10]:
ss,sss

(0.9181841831425598, 0.7118844551408738)

In [ ]:
sns.heatmap(missing_value[missing_value>.5].sort_values(ascending=False).to_frame())

In [ ]:
df.corr(numeric_only=True)["TARGET"].sort_values(ascending=False).values < 0

In [ ]:
import os 
